# 03 — Synthetic data: true conditional coverage under heteroskedasticity and covariate shift
Because Y | X is known here, conditional coverage P(Y ∈ C(X) | X = x) is computed exactly (by Monte
Carlo) for every test input, instead of the WSC / CEC-X proxies used on real data.

**Design.** Following Gan & Liu (2025, Sec. 4): X ∈ R⁵, X ~ N(0, 2² I) for training; under shift the
test inputs are X ~ N(c·(1, 1, −1, −1, 0), 2² I) — Gan & Liu's Scenario Ic is c = 1;
**[my choice]** c ∈ {0, 0.5, 1, 1.5} to vary the shift. The density ratio is then known exactly.

* **S1 (Gan & Liu Scenario I)**: Y = Xβ + ε, ε ~ N(0, I₂), β entries ~ U(0, 1) redrawn each replicate.
* **S2 [my choice], heteroskedastic and correlated noise**: mean of Gan & Liu Scenario II,
  μ₁ = logistic(β₁ᵀX), μ₂ = β₂₁X₁² + β₂₂X₂² + β₂₃X₃² + β₂₄X₄ + β₂₅X₅; noise ε = A(X)B, B ~ N(0, I₂),
  with scales s₁ = 0.5 + 0.5|X₁|, s₂ = 0.5 + 0.5|X₂| and correlation ρ = 0.8 tanh(X₃/2).
  The shift moves X₁, X₂ up and X₃ down, so the noise gets larger and its correlation changes sign.

**Metrics** (per replicate, then averaged): marginal coverage; mean |conditional coverage − 0.9|;
% of test inputs with conditional coverage < 80%; median region area.

**Weight clipping (`RATIO_CLIP`)** and **weighted OTCP / GRCP** (added after notebooks 06 and 08):
with `RATIO_CLIP = 0.99` the estimated ratio is capped at the 99% quantile of the training weights for
ALL weighted methods **[my choice]**; results go to `results/synthetic_clip99/`. The true-ratio
(oracle) rows are NOT clipped. `RATIO_CLIP = None` reproduces the original run in `results/synthetic/`
(plus the two weighted OTCP / GRCP rows).

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
import cpshift as cp
import experiment as ex
import deep

print("device:", deep.DEVICE)

## Configuration

In [ ]:
QUICK = True
RATIO_CLIP = 0.99      # None = no clipping (original run)

CFG = dict(
    SEED=2026, ALPHA=0.10, N_REPS=20, B=1000, SCP_FIT_FRAC=0.5,
    N_TRAIN=1000,          # [my choice] as for Parkinson's (Gan & Liu use 500; the network needs more)
    N_UNLABELED=1000,      # [my choice] unlabeled test inputs used to estimate the density ratio
    N_TEST_X=300,          # [my choice] test inputs at which conditional coverage is evaluated
    M_Y=200,               # [my choice] Monte Carlo draws of Y | X per test input
    SHIFTS=[0.0, 0.5, 1.0, 1.5],
    DEEP_CAL_FRAC=0.5, DEEP_VAL_FRAC=0.2, HIDDEN=64, N_LAYERS=2, LR=1e-3, WEIGHT_DECAY=1e-4,
    BATCH=64, MAX_EPOCHS=500, PATIENCE=30, EPS=1e-3,
    OT_M=1024, OT_EPS=0.1, GRID_SIDE=150, GRCP_VOL_POINTS=50,
)
if QUICK:
    CFG.update(N_REPS=1, B=200, MAX_EPOCHS=40, SHIFTS=[0.0, 1.5], N_TEST_X=100, M_Y=100, GRCP_VOL_POINTS=10)

OUT_DIR = "../results/synthetic" + (f"_clip{round(100 * RATIO_CLIP)}" if RATIO_CLIP else "")
os.makedirs(OUT_DIR, exist_ok=True)
MU0 = np.array([1.0, 1.0, -1.0, -1.0, 0.0])   # Gan & Liu Scenario Ic direction
SD_X = 2.0

## Data-generating processes

In [ ]:
def draw_beta(rng):
    return rng.uniform(0, 1, size=(5, 2))

def mean_fn(X, beta, scenario):
    if scenario == 'S1':
        return X @ beta
    mu1 = 1 / (1 + np.exp(-(X @ beta[:, 0])))
    b = beta[:, 1]
    mu2 = b[0] * X[:, 0] ** 2 + b[1] * X[:, 1] ** 2 + b[2] * X[:, 2] ** 2 + b[3] * X[:, 3] + b[4] * X[:, 4]
    return np.c_[mu1, mu2]

def noise(X, rng, scenario):
    B = rng.normal(size=(len(X), 2))
    if scenario == 'S1':
        return B
    s1, s2 = 0.5 + 0.5 * np.abs(X[:, 0]), 0.5 + 0.5 * np.abs(X[:, 1])
    rho = 0.8 * np.tanh(X[:, 2] / 2)
    return np.c_[s1 * B[:, 0], s2 * (rho * B[:, 0] + np.sqrt(1 - rho ** 2) * B[:, 1])]

def draw_X(n, c, rng):
    return c * MU0 + SD_X * rng.normal(size=(n, 5))

def oracle_ratio(c):
    m = c * MU0
    return lambda X: np.exp(X @ m / SD_X ** 2 - m @ m / (2 * SD_X ** 2))

## Run

In [ ]:
NAMES = ex.METHODS_ALL + ['Deep-LocalCov + WCP, oracle r', 'Deep-LocalCov + EUc, oracle r']
a = CFG['ALPHA']
rows, t0 = [], time.time()
for scenario in ['S1', 'S2']:
    for rep in range(CFG['N_REPS']):
        rng = np.random.default_rng(CFG['SEED'] + rep + (0 if scenario == 'S1' else 10_000))
        beta = draw_beta(rng)
        Xtr = draw_X(CFG['N_TRAIN'], 0.0, rng)
        Ytr = mean_fn(Xtr, beta, scenario) + noise(Xtr, rng, scenario)
        fm = ex.FittedMethods(Xtr, Ytr, CFG, rng, deep, CFG['SEED'] + rep)
        for c in CFG['SHIFTS']:
            Xu = draw_X(CFG['N_UNLABELED'], c, rng)                 # unlabeled test inputs
            r_hat = cp.fit_density_ratio(Xtr, Xu)
            if RATIO_CLIP:
                r_hat = cp.clip_ratio(r_hat, Xtr, RATIO_CLIP)
            Xte = draw_X(CFG['N_TEST_X'], c, rng)
            M = CFG['M_Y']
            Xrep = np.repeat(Xte, M, axis=0)
            Yrep = mean_fn(Xrep, beta, scenario) + noise(Xrep, rng, scenario)
            vol_idx = np.arange(CFG['GRCP_VOL_POINTS']) * M          # first draw of the first inputs
            res = fm.evaluate(Xrep, Yrep, r_hat, rng, vol_idx, oracle_r=oracle_ratio(c))
            for m in NAMES:
                cc = res[m]['covered'].reshape(-1, M).mean(1)      # conditional coverage per input
                v = res[m]['vol'] if m in ('GRCP', 'GRCP + WCP') else res[m]['vol'].reshape(-1, M)[:, 0]
                rows.append(dict(scenario=scenario, rep=rep, shift=c, method=m,
                                 coverage=100 * cc.mean(),
                                 cond_dev=100 * np.mean(np.abs(cc - (1 - a))),
                                 pct_below_80=100 * np.mean(cc < 0.8),
                                 volume_median=np.median(v),
                                 pct_infinite=100 * np.mean(~np.isfinite(v))))
        print(f"{scenario} rep {rep + 1}/{CFG['N_REPS']} | elapsed {time.time() - t0:.0f}s")
res_df = pd.DataFrame(rows)
res_df.to_csv(f"{OUT_DIR}/results_all_reps.csv", index=False)

## Tables

In [ ]:
summ = (res_df.groupby(['scenario', 'method', 'shift'])
        .agg(coverage=('coverage', 'mean'), cond_dev=('cond_dev', 'mean'),
             pct_below_80=('pct_below_80', 'mean'), volume=('volume_median', 'median'),
             pct_infinite=('pct_infinite', 'mean'))
        .round(2).reset_index())
summ.to_csv(f"{OUT_DIR}/summary.csv", index=False)
for sc in ['S1', 'S2']:
    for col in ['coverage', 'cond_dev', 'pct_below_80', 'volume']:
        tab = summ[summ.scenario == sc].pivot(index='method', columns='shift', values=col).reindex(NAMES)
        tab.to_csv(f"{OUT_DIR}/table_{sc}_{col}.csv")
        print(f"\n{sc} — {col} by shift c\n", tab.to_string())

## Figure: conditional-coverage error and coverage versus shift (S2)

In [ ]:
main = ['EU', 'EUc-LR', 'WCP-L2-LR', 'OTCP', 'GRCP', 'OTCP + WCP', 'Deep-LocalCov',
        'Deep-LocalCov + WCP (ours)', 'Deep-LocalCov + EUc (ours)']
fig, ax = plt.subplots(2, 3, figsize=(13, 7))
for r, sc in enumerate(['S1', 'S2']):
    s0 = summ[summ.scenario == sc]
    for m in main:
        s = s0[s0.method == m]
        ls = '-' if ('ours' in m or m in ('EUc-LR', 'WCP-L2-LR')) else '--'
        ax[r, 0].plot(s['shift'], s.coverage, marker='o', ls=ls, label=m if r == 0 else None)
        ax[r, 1].plot(s['shift'], s.cond_dev, marker='o', ls=ls)
        ax[r, 2].plot(s['shift'], s.volume, marker='o', ls=ls)
    ax[r, 0].axhline(90, color='k', lw=1, ls=':')
    ax[r, 0].set_ylabel(f'{sc}: coverage (%)'); ax[r, 1].set_ylabel('mean |cond. cov − 0.9| (%)')
    ax[r, 2].set_ylabel('median region area'); ax[r, 2].set_yscale('log')
for x in ax[-1]:
    x.set_xlabel('shift $c$')
fig.legend(loc='lower center', ncol=4, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_synthetic.png", dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
print("Files saved in", os.path.abspath(OUT_DIR), ":", sorted(os.listdir(OUT_DIR)))
print("CONFIG:", CFG, "| RATIO_CLIP:", RATIO_CLIP)